# 📊 03 · Dashboards com IA (AI/BI Dashboards + Genie Code)

**Objetivo:** construir um dashboard executivo da TelcoBR **só pedindo em português** para o Genie Code — sem SQL, sem arrastar campo por campo.

⏱️ **Duração:** 40–50 min

### 🗺️ O que você vai construir
| Página | Pergunta de negócio que ela responde |
|---|---|
| **1. Visão Executiva** | Como está a receita, a base de clientes e a satisfação? |
| **2. Retenção** | Quem está cancelando, por quê, e o que aconteceu depois do reajuste de mar/2026? |
| **3. Atendimento** | Quais canais encantam e quais irritam o cliente? |
| **4. Lojas e Metas** | Quais lojas estão batendo a meta? (usa o resultado do exercício de Data Prep) |

### 📦 Tabelas disponíveis (no catálogo/schema que você usou no notebook 01)
`clientes` · `contratos` · `planos` · `lojas` · `base_mensal` · `faturamento_mensal` · `atendimentos` · `vendas_lojas_tratadas`

### 📏 Definições oficiais (use sempre estas!)
| KPI | Como a TelcoBR calcula |
|---|---|
| **Receita** | Soma de `valor_faturado` em `faturamento_mensal` |
| **Base ativa** | Contratos com `ativo_inicio_mes = true` em `base_mensal` |
| **Churn mensal** | Contratos com `cancelado_no_mes` ÷ contratos com `ativo_inicio_mes` (no mesmo mês) |
| **ARPU** | Receita do mês ÷ número de **clientes distintos** faturados no mês |
| **NPS** | % de notas 9–10 (promotores) **menos** % de notas 0–6 (detratores). **Não é a média das notas!** |
| **Inadimplência** | Valor faturado com mais de 30 dias de atraso ÷ valor faturado total |

### ⚙️ Configuração — rode esta célula primeiro (ou clique em **Executar tudo**)
Use o **mesmo Catálogo e Schema** do notebook 01 (campos no topo do notebook; padrão no Free Edition: `workspace` / `workshop_negocio`).
As caixas roxas 📋 ao longo do notebook mostram os prompts **já com o nome das suas tabelas** — é só copiar e colar.

In [0]:
import html
dbutils.widgets.text("catalogo", "workspace", "Catálogo")
dbutils.widgets.text("schema", "workshop_negocio", "Schema")
CATALOGO, SCHEMA = dbutils.widgets.get("catalogo").strip(), dbutils.widgets.get("schema").strip()
T = f"{CATALOGO}.{SCHEMA}"
spark.sql(f"USE CATALOG `{CATALOGO}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

def prompt(texto, titulo="📋 Copie e cole"):
    """Mostra um prompt/SQL pronto para copiar, trocando {T} por catálogo.schema."""
    corpo = html.escape(texto.strip().replace("{T}", T))
    displayHTML(f"""<div style="font-family:sans-serif;font-size:12px;font-weight:bold;color:#6929C4;margin:2px 0">{titulo}</div>
<pre style="background:#F4F0FB;border-left:4px solid #6929C4;padding:10px 14px;white-space:pre-wrap;font-size:13px;margin:0">{corpo}</pre>""")

prompt(f"Catálogo: {CATALOGO}\nSchema:   {SCHEMA}\nSuas tabelas: {T}.<tabela>", "✅ Configuração ativa")

---
## 🚀 Passo 1 · Crie o dashboard e conecte os dados

1. Menu lateral → **Dashboards** → **Criar dashboard**. Renomeie para **`TelcoBR Executivo - <seu nome>`**.
2. Abra a aba **Dados** → **+ Adicionar fonte de dados** → navegue até o **seu catálogo → `workshop_negocio`** (veja a caixa ✅ Configuração ativa acima) e selecione:
   `faturamento_mensal`, `base_mensal`, `contratos`, `clientes`, `planos`, `atendimentos`, `vendas_lojas_tratadas`.
3. Volte para a aba da **página** (canvas) e abra o **Genie Code** (ícone ✨). Todos os prompts abaixo vão nele.

> 💡 **Como escrever bons prompts para dashboards**
> - Diga **o tipo de gráfico**, **a métrica**, **o eixo/quebra** e **o filtro**.
> - Repita a **definição do KPI** quando ela não for óbvia (ex.: NPS, churn). A IA não adivinha regra de negócio!
> - Peça um visual por vez e confira antes de seguir.
> - Gostou mas quer ajustar? *"No último gráfico, troque para barras horizontais e ordene do maior para o menor."*

---
## 📄 Página 1 · Visão Executiva

**Prompt 1.1 — Criar a base de dados mensal**
```
Crie um dataset chamado "kpis_mensais" com uma linha por mês (mes_referencia) contendo:
- receita = soma de valor_faturado da tabela faturamento_mensal
- clientes_faturados = número de cliente_id distintos em faturamento_mensal
- arpu = receita / clientes_faturados
- base_ativa = quantidade de contratos com ativo_inicio_mes verdadeiro na tabela base_mensal
- cancelamentos = quantidade de contratos com cancelado_no_mes verdadeiro na base_mensal
- churn = cancelamentos / base_ativa
```

**Prompt 1.2 — Contadores (KPIs) do topo**
```
Usando o dataset kpis_mensais, crie 4 contadores lado a lado no topo da página, considerando apenas o último mês disponível:
Receita do mês (em R$), Base ativa, ARPU (em R$) e Churn (em %).
```

**Prompt 1.3 — Tendência de receita**
```
Crie um gráfico de linha com a receita por mês, com o título "Evolução da receita mensal".
```

**Prompt 1.4 — Receita por categoria**
```
Crie um gráfico de barras empilhadas com a receita mensal por categoria de plano
(junte faturamento_mensal com planos pelo plano_id).
```

**Prompt 1.5 — Receita por região**
```
Crie um gráfico de barras horizontais com a receita de 2026 por região do cliente
(junte faturamento_mensal com clientes pelo cliente_id), ordenado da maior para a menor.
```

**Prompt 1.6 — Filtros**
```
Adicione filtros de região e de categoria de plano no topo da página que se apliquem a todos os gráficos.
```

### ✅ Confira os números da Página 1
Rode a célula abaixo — seus contadores devem mostrar os mesmos valores do último mês.

In [0]:
%sql
WITH fat AS (
  SELECT mes_referencia, sum(valor_faturado) AS receita, count(DISTINCT cliente_id) AS clientes_faturados
  FROM faturamento_mensal GROUP BY ALL
), base AS (
  SELECT mes_referencia, count_if(ativo_inicio_mes) AS base_ativa, count_if(cancelado_no_mes) AS cancelamentos
  FROM base_mensal GROUP BY ALL
)
SELECT mes_referencia,
       format_number(receita, 2) AS receita_rs,
       base_ativa,
       round(receita / clientes_faturados, 2) AS arpu_rs,
       round(100 * cancelamentos / base_ativa, 2) AS churn_pct
FROM fat JOIN base USING (mes_referencia)
ORDER BY mes_referencia DESC

---
## 📄 Página 2 · Retenção
Crie uma nova página (botão **+** ao lado das abas das páginas) chamada **Retenção** e continue no Genie Code:

**Prompt 2.1 — Churn por categoria ao longo do tempo**
```
Crie um gráfico de linhas com o churn mensal por categoria de plano.
Churn = contratos com cancelado_no_mes verdadeiro dividido por contratos com ativo_inicio_mes verdadeiro, na tabela base_mensal,
juntando com planos pelo plano_id. Mostre em percentual.
```
👀 *Repare no que acontece a partir de mar/2026…*

**Prompt 2.2 — Por que os clientes saem?**
```
Crie um gráfico de barras com a quantidade de contratos cancelados por motivo_cancelamento (tabela contratos),
comparando dois períodos: "Antes do reajuste" (cancelamento antes de 01/03/2026) e "Depois do reajuste".
```

**Prompt 2.3 — Churn por região**
```
Crie um mapa ou gráfico de barras com o churn médio mensal de 2026 por região do cliente.
```

**Prompt 2.4 — Qualidade das vendas por canal**
```
Para os contratos ativados a partir de 2025, calcule o percentual que foi cancelado em até 90 dias após a ativação,
por tipo de canal da loja que vendeu (junte contratos com lojas pelo loja_id). Mostre em barras.
```
💬 *Qual canal vende "cliente que não fica"?*

### ✅ Confira a Página 2

In [0]:
%sql
SELECT l.tipo_canal,
       count(*) AS contratos_ativados,
       round(100 * count_if(datediff(c.data_cancelamento, c.data_ativacao) <= 90) / count(*), 1) AS pct_cancelado_90_dias
FROM contratos c
JOIN lojas l USING (loja_id)
WHERE c.data_ativacao >= '2025-01-01' AND c.data_ativacao < '2026-06-01'
GROUP BY ALL ORDER BY pct_cancelado_90_dias DESC

---
## 📄 Página 3 · Atendimento

**Prompt 3.1 — NPS por canal (atenção à definição!)**
```
Crie um gráfico de barras com o NPS por canal_atendimento da tabela atendimentos.
NPS = percentual de notas 9 e 10 menos o percentual de notas de 0 a 6, multiplicado por 100. Não use a média das notas.
Ordene do maior para o menor.
```
🧪 **Experimento:** peça também *"Crie um gráfico com a média da nota_nps por canal"* e compare. A média "esconde" o tamanho do problema da URA!

**Prompt 3.2 — Volume por motivo ao longo do tempo**
```
Crie um gráfico de área empilhada com a quantidade de atendimentos por mês e por motivo_contato.
```

**Prompt 3.3 — Resolver de primeira faz diferença?**
```
Crie uma tabela comparando atendimentos resolvidos e não resolvidos no primeiro contato:
quantidade, tempo médio de resolução em horas e NPS (mesma definição de antes).
```

**Prompt 3.4 — Mapa de calor**
```
Crie um mapa de calor com canal_atendimento nas linhas, motivo_contato nas colunas e o NPS como cor.
```

### ✅ Confira a Página 3 — NPS correto × média das notas

In [0]:
%sql
SELECT canal_atendimento,
       round(100 * (count_if(nota_nps >= 9) - count_if(nota_nps <= 6)) / count(*), 1) AS nps_correto,
       round(avg(nota_nps), 2) AS media_das_notas_errado,
       round(100 * avg(cast(resolvido_primeiro_contato AS INT)), 1) AS pct_resolvido_1o_contato
FROM atendimentos
GROUP BY ALL ORDER BY nps_correto DESC

---
## 📄 Página 4 · Lojas e Metas
Use a tabela `vendas_lojas_tratadas` (gabarito) ou a sua `vendas_tratadas_<seu_nome>` do exercício 02.

**Prompt 4.1**
```
Usando vendas_lojas_tratadas, crie um contador com o percentual de lojas-mês que bateram a meta (bateu_meta) em 2026.
```
**Prompt 4.2**
```
Crie uma tabela com o ranking das 15 lojas com maior atingimento médio de meta em 2026,
mostrando nome_loja, cidade, tipo_canal, receita total e atingimento médio em %. Destaque em vermelho quem ficou abaixo de 100%.
```
**Prompt 4.3**
```
Crie um gráfico de barras com a receita de vendas versus a meta de receita por mês, com as duas séries lado a lado.
```

---
## ✨ Passo final · Publicar e compartilhar
1. Clique em **Publicar** — escolha *credenciais compartilhadas* para que o público veja sem precisar de acesso às tabelas.
2. **Compartilhar** → adicione colegas ou o grupo do time.
3. **Agendar** → envio por e-mail toda segunda às 8h (PDF do dashboard).
4. Ative o **Genie** no dashboard publicado (*Ask Genie*) — assim o leitor pode fazer perguntas em linguagem natural sobre os dados do próprio dashboard.

### 🛟 Plano B · Dashboard gabarito
Travou ou quer comparar? Rode o notebook **`06_gabaritos_plano_b`**: ele cria na sua conta um dashboard de referência completo (feito sobre as Metric Views oficiais do notebook 05).

### 🏆 Desafios extras
```
Escreva um texto curto no topo da página Retenção resumindo os 3 principais insights dos gráficos.
```
```
Aplique uma paleta de cores com roxo como cor principal em todos os gráficos.
```
```
Crie um gráfico de dispersão: score_credito médio × inadimplência por faixa_etaria.
Inadimplência = valor faturado com dias_atraso > 30 dividido pelo valor faturado total.
```

### 💬 Para discutir
- Onde a IA acertou de primeira? Onde precisou da **definição do KPI** para acertar?
- E se todo mundo da empresa usasse a **mesma** definição de churn e NPS automaticamente? → É isso que veremos com **Genie Agents** (notebook 04) e com a **camada semântica / Metric Views** (notebook 05).